In [ ]:
%load_ext autoreload
%autoreload 2

# Cardiac Imaging in Medicare — main driver

Thin driver: all logic lives in `src/cardiac_imaging/`. Fix bugs there, not in this notebook.

Run headless (Claude Code does this in the debug loop):

```bash
papermill notebooks/main.ipynb runs/out.ipynb
```

Each run is saved to its own folder in `~/Downloads/cms_runs/` (linked as `runs/` in the repo) with a `gallery.html` of every chart; `runs/index.html` lists all runs.

Data: local DuckDB `~/Downloads/cms_parquet/cms.duckdb` (claims view over Parquet + reference tables), opened read-only.

In [ ]:
# Parameters — papermill overrides these, e.g.
#   papermill notebooks/main.ipynb runs/out.ipynb -p LABEL my-note -p SAVE_TIFF False
# DATABASE: None uses the config default (~/Downloads/cms_parquet/cms.duckdb or $CARDIAC_DB)
# REPORT_ERRORS: write logs/last_error.md on failure (+ GitHub issue if CARDIAC_GITHUB_TOKEN is set)
# LABEL: optional note added to the run folder name
# SAVE_TIFF: False skips the submission TIFFs (faster, ~10x smaller run folder)
DATABASE = None
REPORT_ERRORS = True
LABEL = None
SAVE_TIFF = True

In [ ]:
# Find the repo, load .env, make src/ importable
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
env_file = ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            k, v = line.split('=', 1)
            if v.strip():
                os.environ.setdefault(k.strip(), v.strip())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

In [ ]:
# Connect read-only and check row counts per year
from cardiac_imaging.db.connection import get_connection, require_tables
con = get_connection(DATABASE)
require_tables(con)
con.sql("SELECT year, COUNT(*) AS rows FROM partb_provider_service GROUP BY year ORDER BY year").show()

In [ ]:
# Run the full analysis → a new run folder with figures, tables, log, gallery.html and all_figures.pdf
from cardiac_imaging import config, pipeline
config.SAVE_TIFF = SAVE_TIFF
ctx = pipeline.run_all(con=con, report_errors=REPORT_ERRORS, label=LABEL)

In [ ]:
# Where this run's outputs are
print('Gallery :', ctx['run_dir'] / 'gallery.html')
print('PDF     :', ctx['run_dir'] / 'all_figures.pdf')
print('All runs:', config.RUNS_DIR / 'index.html')

Re-run one step after a fix (reuses earlier results in `ctx`): `dict(pipeline.STEPS)['figures3_7_state_maps'](ctx)`